# Colab training from prepared SynthHuman shards

The active experiment uses 50,000 samples, depth + normals, 256×256 inputs, and 5 epochs. Run prepare_synthhuman_shards.ipynb first: benchmark two shards (500 preparation samples), then continue the full selection. This training notebook requires the completed shard manifest; it does not call the source-file Drive API during training. A bounded local cache receives archives from a small Drive folder. Sample order is shuffled by shard and within each shard to avoid copying a different archive for every sample.

In [ ]:
# 0. Dependency preflight (run once per fresh Colab runtime)
import importlib.util
missing = [name for name in ('timm', 'skimage', 'cv2') if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError('Missing packages: ' + ', '.join(missing) + '. In a separate Colab cell run: %pip install timm scikit-image opencv-python-headless, then restart the runtime and rerun setup.')
print('Dependencies present: timm, scikit-image, and OpenCV (cv2; EXR-capable Colab build)')


In [ ]:
# 1. Runtime, imports, and experiment settings
# Run this after connecting the Colab runtime. Drive is mounted in the next cell.
import os, json, math, random, time, platform
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt
try:
    from skimage.metrics import structural_similarity as skimage_ssim
except ImportError:
    skimage_ssim = None

SEED = 42
INPUT_SIZE = (256, 256)  # H, W; fixed shape for initial Core ML conversion
BATCH_SIZE, EPOCHS, NUM_WORKERS = 8, 5, 2
DATASET_MODE = 'pilot'  # This notebook is set up for a bounded 50k-sample experiment.
PILOT_SAMPLE_COUNT = 50_000
EXPECTED_SYNTHHUMAN_SAMPLES = 298008
LR, WEIGHT_DECAY, VAL_FRACTION, TEST_FRACTION = 1e-4, 1e-4, 0.15, 0.15
DEPTH_MODE = 'metric'  # SynthHuman absolute z-depth converted from centimetres to metres
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = torch.cuda.is_available()
print({'device':str(DEVICE),'torch':torch.__version__,'python':platform.python_version(),'input_size':INPUT_SIZE})
if torch.cuda.is_available():
    print({'cuda_available':True,'cuda_version':torch.version.cuda,'cudnn_version':torch.backends.cudnn.version(),'gpu':torch.cuda.get_device_name(0),'gpu_count':torch.cuda.device_count(),'gpu_capability':torch.cuda.get_device_capability(0),'gpu_memory_GiB':round(torch.cuda.get_device_properties(0).total_memory/1024**3,2)})
else:
    print({'cuda_available':False,'message':'Training will run on CPU; in Colab choose Runtime > Change runtime type > GPU.'})

### Google Drive persistence
Colab runtimes are temporary. Mount Drive at the start of every new runtime session, then keep the dataset, manifest, pretrained-weight caches, checkpoints, training history, metrics, and Core ML exports under `MyDrive/AppleCapstoneRelighting`.

In [ ]:
# 2. Mount Google Drive and configure persistent paths
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/AppleCapstoneRelighting')
DATA_ROOT = DRIVE_ROOT / 'dataset'
if DATASET_MODE == 'pilot':
    MANIFEST = DATA_ROOT / 'pilot_manifests' / f'synthhuman_{PILOT_SAMPLE_COUNT}_seed{SEED}.jsonl'
    OUTPUT_DIR = DRIVE_ROOT / 'runs' / f'fastvit_t8_fpn_depth_normals_pilot{PILOT_SAMPLE_COUNT}_seed{SEED}_5ep_driveapi_v2'
else:
    MANIFEST = DATA_ROOT / 'manifest.jsonl'
    OUTPUT_DIR = DRIVE_ROOT / 'runs' / 'fastvit_t8_fpn_depth_normals_full_5ep_v1'
CACHE_ROOT = DRIVE_ROOT / 'cache'

# Keep downloaded model assets and experiment outputs between Colab sessions.
os.environ['TORCH_HOME'] = str(CACHE_ROOT / 'torch')
os.environ['HF_HOME'] = str(CACHE_ROOT / 'huggingface')
for folder in (DATA_ROOT, OUTPUT_DIR, CACHE_ROOT / 'torch', CACHE_ROOT / 'huggingface'):
    folder.mkdir(parents=True, exist_ok=True)

assert Path('/content/drive/MyDrive').is_dir(), 'Google Drive is not mounted.'
print('Persistent Drive root:', DRIVE_ROOT)
print('Dataset root:', DATA_ROOT)
print('Manifest:', MANIFEST)
print('Run outputs:', OUTPUT_DIR)
print('Run ID/config folder:', OUTPUT_DIR.name)
print('This run reads selected samples from Drive and writes checkpoints/logs to Drive.')

### Prepared dataset preflight

The next cell checks the completed packed-data manifest. A 500-sample preparation benchmark is intentionally insufficient for this 50k run. Complete preparation by setting PREPARE_SHARD_LIMIT=None in the preparation notebook. Drive retains the archive copies and checkpoints; the local cache can be rebuilt after a runtime reset.

In [ ]:
# 2a. Load the prepared manifest; no source dataset FUSE/API reads here.
SHARD_ROOT=DATA_ROOT/'prepared_shards'/f'synthhuman_{PILOT_SAMPLE_COUNT}_seed{SEED}_{INPUT_SIZE[0]}x{INPUT_SIZE[1]}_v1'
SHARD_MANIFEST_PATH=SHARD_ROOT/'manifest.json'
if not SHARD_MANIFEST_PATH.is_file():
    raise FileNotFoundError('Run prepare_synthhuman_shards.ipynb first to pack the existing dataset.')
PACKED_MANIFEST=json.loads(SHARD_MANIFEST_PATH.read_text())
config=PACKED_MANIFEST['config']
if not PACKED_MANIFEST.get('complete'):
    ready=sum(len(s['rows']) for s in PACKED_MANIFEST['shards'])
    raise RuntimeError(f'Only {ready:,} samples are packed so far. In prepare_synthhuman_shards.ipynb, set PREPARE_SHARD_LIMIT=None and rerun preparation to finish the full selection.')
if config['samples']!=PILOT_SAMPLE_COUNT or config['input_size']!=list(INPUT_SIZE) or config['seed']!=SEED:
    raise ValueError('Prepared dataset configuration does not match the training settings.')
DATA_IO_MODE='drive_shards'
OUTPUT_DIR=DRIVE_ROOT/'runs'/f'fastvit_t8_fpn_depth_normals_shards{PILOT_SAMPLE_COUNT}_seed{SEED}_v1'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
print('Prepared archives:',len(PACKED_MANIFEST['shards']),'| samples:',config['samples'])
print('Training outputs:',OUTPUT_DIR)

In [ ]:
# 3. Validate archive metadata and build the training manifest.
manifest_rows=[]
seen_ids=set()
for record in PACKED_MANIFEST['shards']:
    if Path(record['name']).name!=record['name']:
        raise ValueError('Invalid shard filename in manifest.')
    path=SHARD_ROOT/record['name']
    if not path.is_file() or path.stat().st_size!=record['bytes']:
        raise IOError(f'Missing/incomplete prepared archive: {path}. Rerun preparation to repair it.')
    for row in record['rows']:
        if row['subject_id'] in seen_ids or row['partition']!=record['partition'] or row['shard']!=record['name']:
            raise ValueError('Duplicate or mismatched sample assignment in packed manifest.')
        seen_ids.add(row['subject_id'])
        manifest_rows.append(row)
if len(manifest_rows)!=PILOT_SAMPLE_COUNT:
    raise ValueError(f'Expected {PILOT_SAMPLE_COUNT:,} samples; manifest has {len(manifest_rows):,}.')
MANIFEST=DATA_ROOT/'pilot_manifests'/f'synthhuman_shards_{PILOT_SAMPLE_COUNT}_seed{SEED}_v1.jsonl'
MANIFEST.parent.mkdir(parents=True,exist_ok=True)
temporary=MANIFEST.with_suffix('.jsonl.tmp')
temporary.write_text(''.join(json.dumps(r)+'\n' for r in manifest_rows))
temporary.replace(MANIFEST)
print('Prepared sample manifest:',MANIFEST)

### Manifest validation
Provide one JSONL row per labeled sample. In addition to file paths and `subject_id`, each row records `label_source`, `depth_units`, `normal_convention`, so the depth and normal targets remain auditable.

In [ ]:
# 4. Read and validate the automatically generated manifest
def load_manifest(path,verify_paths=False):
    if not path.is_file(): raise FileNotFoundError(f'Manifest missing: {path}. Run the dataset manifest cell first.')
    rows=[json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    required={'image','depth','normal','mask','subject_id','label_source','depth_units','normal_convention'}
    if not rows: raise ValueError('Manifest contains no rows')
    for i,row in enumerate(rows):
        missing=required-row.keys()
        if missing: raise ValueError(f'row {i} missing fields: {sorted(missing)}')
        for key in ('image','depth','normal','mask'):
            p=Path(row[key]); p=p if p.is_absolute() else DATA_ROOT/p
            if verify_paths and not p.is_file(): raise FileNotFoundError(f'row {i} {key}: {p}')
            row[key]=str(p)
    return rows
ALL_ROWS=load_manifest(MANIFEST,verify_paths=False)
if DATASET_MODE == 'full':
    rows=ALL_ROWS
    RUN_MANIFEST=MANIFEST
elif DATASET_MODE == 'pilot':
    if len(ALL_ROWS)<PILOT_SAMPLE_COUNT:
        raise ValueError(f'Pilot requires {PILOT_SAMPLE_COUNT} complete labeled samples, but manifest has {len(ALL_ROWS)}.')
    pilot_rng=random.Random(SEED)
    rows=sorted(pilot_rng.sample(ALL_ROWS,PILOT_SAMPLE_COUNT),key=lambda r:r['subject_id'])
    RUN_MANIFEST=DATA_ROOT/'pilot_manifests'/f'depth_pilot_{PILOT_SAMPLE_COUNT}_seed{SEED}.jsonl'
    RUN_MANIFEST.parent.mkdir(parents=True,exist_ok=True)
    pilot_tmp=RUN_MANIFEST.with_suffix('.jsonl.tmp'); pilot_tmp.write_text(''.join(json.dumps(r)+'\n' for r in rows)); pilot_tmp.replace(RUN_MANIFEST)
else:
    raise ValueError("DATASET_MODE must be 'full' or 'pilot'")
print(f'Run mode={DATASET_MODE}; samples={len(rows):,}; active labels: depth + normals + mask')
print('Run manifest:',RUN_MANIFEST)

### Identity split
Generate the fixed train/validation/test split before augmentations. The saved `split.json` makes the partition reviewable and reproducible.

In [ ]:
# 5. Split by sample identity (SynthHuman release has no subject-ID field)
def split_by_subject(rows,seed=SEED):
    subjects=sorted({r['subject_id'] for r in rows})
    if len(subjects)<5:
        parts={'train':rows,'val':[],'test':[]}
        print(f'AUDIT MODE: only {len(subjects)} sample(s). This permits the loader audit only; training is blocked until a chunk is downloaded and at least five sample IDs are available.')
        return parts
    rng=random.Random(seed); rng.shuffle(subjects)
    n_test=max(1,round(len(subjects)*TEST_FRACTION)); n_val=max(1,round(len(subjects)*VAL_FRACTION))
    if n_test+n_val>=len(subjects): raise ValueError('Not enough samples for requested split fractions')
    test_ids=set(subjects[:n_test]); val_ids=set(subjects[n_test:n_test+n_val]); train_ids=set(subjects)-test_ids-val_ids
    parts={name:[r for r in rows if r['subject_id'] in ids] for name,ids in [('train',train_ids),('val',val_ids),('test',test_ids)]}
    split_info={'seed':seed,'split_unit':'sample_id (not verified person identity)','train_subjects':sorted(train_ids),'val_subjects':sorted(val_ids),'test_subjects':sorted(test_ids)}
    (OUTPUT_DIR/'split.json').write_text(json.dumps(split_info,indent=2))
    return parts
parts=split_by_subject(rows)
if any(r['partition'] != name for name, group in parts.items() for r in group):
    raise ValueError('Split differs from the prepared partitions. Use the same seed and split fractions as preparation.')
print({k:len(v) for k,v in parts.items()})
print('Evaluation limitation: split is sample-disjoint; the source release does not expose subject IDs for identity-disjoint evaluation.')

### Image and label preprocessing
This cell defines loading/resizing and paired transforms only. It does not download data, change the split, or start training.

In [ ]:
# Reader for prepared shards. Embedded in the training notebook preprocessing cell.
from collections import OrderedDict
import hashlib
import io
import shutil
import tarfile

SHARD_CACHE_GIB = 4
LOCAL_SHARD_CACHE = Path('/content/synthhuman_shard_cache') / PACKED_MANIFEST['config']['selection_sha256'][:16]
LOCAL_SHARD_CACHE.mkdir(parents=True, exist_ok=True)
SHARD_CACHE_BYTES = min(SHARD_CACHE_GIB * 1024**3, max(0, shutil.disk_usage('/content').free - 2*1024**3))
SHARD_RECORDS = {entry['name']: entry for entry in PACKED_MANIFEST['shards']}
SHARD_STATS = {'copies': 0, 'local_hits': 0, 'copied_bytes': 0, 'copy_seconds': 0.0}
for previous_handle in globals().get('shard_handles', {}).values():
    previous_handle.close()
shard_handles = OrderedDict()
shard_cache_bytes = 0
shard_cache_files = OrderedDict()
for existing_path in sorted(LOCAL_SHARD_CACHE.glob('*.tar'), key=lambda x: x.stat().st_mtime):
    record = SHARD_RECORDS.get(existing_path.name)
    if record and existing_path.stat().st_size == record['bytes']:
        shard_cache_files[existing_path.name] = record['bytes']
        shard_cache_bytes += record['bytes']
    else:
        existing_path.unlink()
if max(entry['bytes'] for entry in SHARD_RECORDS.values()) > SHARD_CACHE_BYTES:
    raise OSError('A prepared shard exceeds the available local cache budget.')


def shard_sha256(path):
    result = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(8*1024**2), b''):
            result.update(block)
    return result.hexdigest()


def trim_shard_cache(required_bytes=0):
    global shard_cache_bytes
    while shard_cache_files and shard_cache_bytes + required_bytes > SHARD_CACHE_BYTES:
        oldest, size = shard_cache_files.popitem(last=False)
        handle = shard_handles.pop(oldest, None)
        if handle is not None:
            handle.close()
        (LOCAL_SHARD_CACHE / oldest).unlink(missing_ok=True)
        shard_cache_bytes -= size
trim_shard_cache()


def open_prepared_shard(name):
    global shard_cache_bytes
    if name in shard_handles:
        shard_cache_files.move_to_end(name)
        SHARD_STATS['local_hits'] += 1
        return shard_handles[name]
    record = SHARD_RECORDS[name]
    target = LOCAL_SHARD_CACHE / name
    started = time.perf_counter()
    cached = target.is_file() and target.stat().st_size == record['bytes'] and shard_sha256(target) == record['sha256']
    if not cached:
        if name in shard_cache_files:
            shard_cache_bytes -= shard_cache_files.pop(name)
        target.unlink(missing_ok=True)
        trim_shard_cache(record['bytes'])
        if shutil.disk_usage('/content').free < record['bytes'] + 2*1024**3:
            raise OSError('Insufficient local disk for the next shard. Clear unused local data and rerun.')
        temporary = target.with_suffix('.tar.part')
        try:
            shutil.copyfile(SHARD_ROOT / name, temporary)
            if temporary.stat().st_size != record['bytes'] or shard_sha256(temporary) != record['sha256']:
                raise IOError(f'Shard integrity check failed: {name}')
            temporary.replace(target)
        except Exception:
            temporary.unlink(missing_ok=True)
            raise
        SHARD_STATS['copies'] += 1
        SHARD_STATS['copied_bytes'] += record['bytes']
        SHARD_STATS['copy_seconds'] += time.perf_counter() - started
        print(f'Staged {name}: {record["bytes"]/1024**2:.1f} MiB in {time.perf_counter()-started:.1f}s')
    if name not in shard_cache_files:
        shard_cache_files[name] = record['bytes']
        shard_cache_bytes += record['bytes']
    shard_cache_files.move_to_end(name)
    handle = tarfile.open(target, 'r:')
    handle.getmembers()
    shard_handles[name] = handle
    return handle


class RelightingDataset(Dataset):
    def __init__(self, rows, augment=False):
        self.rows, self.augment = rows, augment
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        row = self.rows[index]
        archive = open_prepared_shard(row['shard'])
        with archive.extractfile(row['member']) as member:
            with np.load(io.BytesIO(member.read()), allow_pickle=False) as sample:
                image = sample['rgb'].astype(np.float32) / 255.0
                depth = sample['depth'].astype(np.float32)
                normal = sample['normal'].astype(np.float32)
                mask = sample['mask'].astype(np.float32)
        expected = {'image': (3,*INPUT_SIZE), 'depth': (1,*INPUT_SIZE), 'normal': (3,*INPUT_SIZE), 'mask': (1,*INPUT_SIZE)}
        for key, array in [('image',image),('depth',depth),('normal',normal),('mask',mask)]:
            if array.shape != expected[key] or not np.isfinite(array).all():
                raise ValueError(f'Invalid prepared {key} for {row["subject_id"]}')
        if mask.sum() <= 0:
            raise ValueError(f'Empty prepared valid mask: {row["subject_id"]}')
        if self.augment and random.random() < .5:
            image,depth,normal,mask = [np.flip(a,axis=2).copy() for a in (image,depth,normal,mask)]
            normal[0] *= -1
        image = (image - np.array([.485,.456,.406],np.float32)[:,None,None]) / np.array([.229,.224,.225],np.float32)[:,None,None]
        tensors = [torch.from_numpy(a.copy()).float() for a in (image,depth,normal,mask)]
        tensors[2] = F.normalize(tensors[2],dim=0,eps=1e-6)
        return tuple(tensors)
print(f'Prepared-shard reader ready. Local cache cap: {SHARD_CACHE_BYTES/1024**3:.2f} GiB.')


def shard_sample_order(dataset, seed):
    grouped = {}
    for index, row in enumerate(dataset.rows):
        grouped.setdefault(row['shard'], []).append(index)
    rng = random.Random(seed)
    names = sorted(grouped)
    rng.shuffle(names)
    order = []
    for name in names:
        rng.shuffle(grouped[name])
        order.extend(grouped[name])
    return order


### What preprocessing produces
Each sample returns four tensors: ImageNet-normalized RGB `(3,256,256)`; metric depth `(1,256,256)` in metres (source EXR centimetres × 0.01); unit normals `(3,256,256)` in camera XYZ; and valid-pixel mask `(1,256,256)`. Invalid/non-finite depth, the EXR sentinel, and zero-length normals are excluded from the mask. The model returns full-resolution depth `(B,1,256,256)` and normals `(B,3,256,256)`. The loader audit prints tensor shapes/ranges and shows one sample before training.


### DataLoader and sample audit
Run this after the previous cells. Inspect the displayed example and ranges before constructing the model.

In [ ]:
# 7. Build loaders and audit one real sample
NUM_WORKERS=0  # One reader owns the local archive cache and tar handles.
datasets={k:RelightingDataset(v,augment=(k=='train')) for k,v in parts.items()}
loaders={k:DataLoader(v,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=(DEVICE.type=='cuda'),drop_last=False) for k,v in datasets.items()}
x,d,n,m=datasets['train'][0]
valid_depth_values=d[0][m[0]>0]
print('Preprocessing output shapes:', {'rgb':tuple(x.shape),'depth_m':tuple(d.shape),'normal_unit':tuple(n.shape),'valid_mask':tuple(m.shape)})
print('RGB normalized channel mean/std:',x.mean((1,2)).tolist(),x.std((1,2)).tolist())
print('valid depth metres p01/p50/p99/max',*[float(torch.quantile(valid_depth_values,q)) for q in (0.01,0.50,0.99)],float(valid_depth_values.max()),'normal range',float(n.min()),float(n.max()),'mask fraction',float(m.mean()))
assert m.sum()>0,'Empty valid mask'
fig,ax=plt.subplots(1,4,figsize=(15,4)); ax[0].imshow(np.clip(x.permute(1,2,0).numpy()*[.229,.224,.225]+[.485,.456,.406],0,1)); ax[0].set_title('RGB'); ax[1].imshow(d[0],cmap='viridis'); ax[1].set_title('Depth (m)'); ax[2].imshow(n.permute(1,2,0)*.5+.5); ax[2].set_title('Normals'); ax[3].imshow(m[0],cmap='gray'); ax[3].set_title('Valid mask'); [z.axis('off') for z in ax]; plt.show()

In [ ]:
# 8. Shared multi-task architecture with a selectable mobile backbone
# Default preserves the project baseline (FastViT-T8). MobileNetV3-Large is an ablation only.
import timm
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights

BACKBONE = 'fastvit_t8.apple_in1k'  # compare candidates via the same train/eval protocol

class FastViTEncoder(nn.Module):
    def __init__(self,pretrained=True):
        super().__init__()
        self.net=timm.create_model('fastvit_t8.apple_in1k',pretrained=pretrained,features_only=True)
        self.channels=tuple(self.net.feature_info.channels())
    def forward(self,x): return self.net(x)

class MobileNetV3Encoder(nn.Module):
    """Experimental alternative; exposes stride-4/8/16/32 features."""
    def __init__(self,pretrained=True):
        super().__init__()
        weights=MobileNet_V3_Large_Weights.DEFAULT if pretrained else None
        self.features=mobilenet_v3_large(weights=weights).features
        self.tap_indices=(3,6,12,15)
        self.channels=(24,40,112,160)
    def forward(self,x):
        outputs=[]
        for idx,layer in enumerate(self.features):
            x=layer(x)
            if idx in self.tap_indices: outputs.append(x)
        if len(outputs)!=len(self.channels): raise RuntimeError('Unexpected MobileNetV3 feature taps')
        return outputs

class DepthwiseSeparableBlock(nn.Module):
    def __init__(self,channels):
        super().__init__()
        self.depthwise=nn.Conv2d(channels,channels,3,padding=1,groups=channels,bias=False)
        self.depthwise_bn=nn.BatchNorm2d(channels)
        self.pointwise=nn.Conv2d(channels,channels,1,bias=False)
        self.pointwise_bn=nn.BatchNorm2d(channels)
    def forward(self,x):
        x=F.relu(self.depthwise_bn(self.depthwise(x)),inplace=True)
        return F.relu(self.pointwise_bn(self.pointwise(x)),inplace=True)

class CompactFPN(nn.Module):
    def __init__(self,channels,width=48):
        super().__init__()
        self.lateral=nn.ModuleList([nn.Conv2d(c,width,1) for c in channels])
        self.smooth=nn.ModuleList([DepthwiseSeparableBlock(width) for _ in channels])
    def forward(self,features):
        y=self.smooth[-1](self.lateral[-1](features[-1]))
        for idx in range(len(features)-2,-1,-1):
            y=F.interpolate(y,size=features[idx].shape[-2:],mode='bilinear',align_corners=False)
            y=self.smooth[idx](y+self.lateral[idx](features[idx]))
        return y

class MobileTaskHead(nn.Module):
    def __init__(self,width,out_channels):
        super().__init__(); self.refine=DepthwiseSeparableBlock(width); self.predict=nn.Conv2d(width,out_channels,1)
    def forward(self,x,out_size):
        return F.interpolate(self.predict(self.refine(x)),size=out_size,mode='bilinear',align_corners=False)

class PortraitGeometryModel(nn.Module):
    """One RGB crop -> metric depth and unit surface normals."""
    def __init__(self,backbone=BACKBONE,pretrained=True,fpn_width=48):
        super().__init__()
        if backbone=='fastvit_t8.apple_in1k': self.encoder=FastViTEncoder(pretrained=pretrained)
        elif backbone=='mobilenet_v3_large': self.encoder=MobileNetV3Encoder(pretrained=pretrained)
        else: raise ValueError(f'Unsupported backbone: {backbone}')
        self.backbone_name=backbone
        self.neck=CompactFPN(self.encoder.channels,width=fpn_width)
        self.depth_head=MobileTaskHead(fpn_width,1)
        self.normal_head=MobileTaskHead(fpn_width,3)
    def forward(self,x):
        out_size=x.shape[-2:]; fused=self.neck(self.encoder(x))
        depth=F.softplus(self.depth_head(fused,out_size))
        normals=F.normalize(self.normal_head(fused,out_size),p=2,dim=1,eps=1e-6)
        return depth,normals

model=PortraitGeometryModel(backbone=BACKBONE,pretrained=True).to(DEVICE)
nparams=sum(p.numel() for p in model.parameters())
with torch.no_grad(): outputs=model(torch.zeros(2,3,*INPUT_SIZE,device=DEVICE))
assert [tuple(t.shape) for t in outputs]==[(2,1,*INPUT_SIZE),(2,3,*INPUT_SIZE)]
assert all(torch.isfinite(t).all() for t in outputs)
print(f'Backbone={BACKBONE}; total parameters={nparams/1e6:.2f}M; FPN width=48; feature channels={model.encoder.channels}')


In [ ]:
# 9. Masked depth and normal losses/metrics
class GeometryObjective(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_vars=nn.Parameter(torch.zeros(2))
    @staticmethod
    def masked_mean(value,mask):
        return (value*mask).sum()/mask.sum().clamp_min(1.)
    def forward(self,pd,td,pn,tn,mask):
        if DEPTH_MODE=='relative':
            pd0=pd.clamp_min(1e-5).log(); td0=td.clamp_min(1e-5).log()
            for b in range(pd.shape[0]):
                valid=mask[b,0].bool()
                if not valid.any(): raise ValueError('A sample has an empty valid mask')
                pd0[b]-=torch.median(pd0[b,0][valid]); td0[b]-=torch.median(td0[b,0][valid])
            depth_loss=self.masked_mean(F.smooth_l1_loss(pd0,td0,reduction='none'),mask)
        else:
            depth_loss=self.masked_mean(F.smooth_l1_loss(pd,td,reduction='none'),mask)
        cosine=(F.normalize(pn,dim=1,eps=1e-6)*F.normalize(tn,dim=1,eps=1e-6)).sum(1,keepdim=True).clamp(-1,1)
        normal_loss=self.masked_mean(1-cosine,mask)
        losses=(depth_loss,normal_loss)
        total=sum(torch.exp(-self.log_vars[j])*loss+self.log_vars[j] for j,loss in enumerate(losses))
        return total,{'depth_loss':depth_loss.detach(),'normal_loss':normal_loss.detach()}

@torch.no_grad()
def evaluate(model,loader):
    model.eval(); sums={}; counts={}
    for x,d,n,m in loader:
        x,d,n,m=[z.to(DEVICE,non_blocking=True) for z in (x,d,n,m)]
        pd,pn=model(x); valid=m[:,0].bool()
        dv=(pd[:,0]-d[:,0])[valid]
        result={'depth_mae_m':dv.abs().mean(),'depth_rmse_m':dv.square().mean().sqrt()}
        p_rel=pd[:,0].clamp_min(1e-5).log(); t_rel=d[:,0].clamp_min(1e-5).log()
        for b in range(x.shape[0]):
            mb=valid[b]
            if not mb.any(): raise ValueError('Evaluation sample has empty mask')
            p_rel[b]-=torch.median(p_rel[b][mb]); t_rel[b]-=torch.median(t_rel[b][mb])
        rr=(p_rel-t_rel)[valid]
        result['depth_log_rmse_median_aligned']=rr.square().mean().sqrt()
        dot=(F.normalize(pn,dim=1)*F.normalize(n,dim=1)).sum(1).clamp(-1,1)
        angle=torch.rad2deg(torch.acos(dot[valid]))
        result.update({'normal_mean_deg':angle.mean(),'normal_median_deg':angle.median(),'normal_within_11_25_deg':(angle<11.25).float().mean()})
        for key,value in result.items():
            sums[key]=sums.get(key,0.)+float(value); counts[key]=counts.get(key,0)+1
    if not counts: raise ValueError('Evaluation partition is empty')
    return {key:sums[key]/counts[key] for key in sums}

objective=GeometryObjective().to(DEVICE)
optimizer=torch.optim.AdamW(list(model.parameters())+list(objective.parameters()),lr=LR,weight_decay=WEIGHT_DECAY)
scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='min',factor=.5,patience=3)
scaler=torch.amp.GradScaler(device='cuda',enabled=(DEVICE.type=='cuda'))
print('Active supervised heads: depth + normals.')


In [ ]:
# 10. Timed training with resumable mid-epoch checkpoints on Drive
CHECKPOINT=OUTPUT_DIR/'best_checkpoint.pt'
LAST=OUTPUT_DIR/'last_checkpoint.pt'
HISTORY_PATH=OUTPUT_DIR/'history.json'
PROGRESS_PATH=OUTPUT_DIR/'progress.json'
CHECKPOINT_EVERY_MINUTES=10  # Full-epoch runs can outlast a Colab session; resume from Drive checkpoint.
PROGRESS_EVERY_STEPS=25  # Persist a compact status record every 25 iterations; tqdm updates every iteration.
print('Persistent checkpoints/history location:', OUTPUT_DIR)
partition_sizes={name:len(group) for name,group in parts.items()}
steps_per_epoch=math.ceil(partition_sizes['train']/BATCH_SIZE)
print(f'Run plan: total={len(rows):,}; split={partition_sizes}; batch={BATCH_SIZE}; epochs={EPOCHS}; train iterations/epoch={steps_per_epoch:,}; max train iterations={steps_per_epoch*EPOCHS:,}')
if any(size==0 for size in partition_sizes.values()):
    raise RuntimeError(f'Cannot train: train/validation/test split is incomplete: {partition_sizes}. Download a full chunk and rerun manifest/split/loader cells first.')

def save_checkpoint(payload,path):
    temp_path=path.with_name(path.name+'.tmp')
    torch.save(payload,temp_path)
    temp_path.replace(path)

def checkpoint_state(epoch,batch_in_epoch,epoch_order,score_value,train_totals=None,task_totals=None,train_seconds_elapsed=0.0):
    return {'epoch':epoch,'batch_in_epoch':batch_in_epoch,'epoch_order':epoch_order,'partial_train_totals':train_totals or [],'partial_task_totals':task_totals or {},'train_seconds_elapsed':train_seconds_elapsed,'model':model.state_dict(),'objective':objective.state_dict(),'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),'best_score':score_value,'history':history,'python_rng':random.getstate(),'numpy_rng':np.random.get_state(),'torch_rng':torch.get_rng_state(),'cuda_rng':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,'config':{'seed':SEED,'input_size':INPUT_SIZE,'depth_mode':DEPTH_MODE,'backbone':BACKBONE,'fpn_width':48,'targets':['depth','normals'],'data_io_mode':DATA_IO_MODE}}

history=[]; start_epoch=0; resume_batch=0; resume_order=None; best_score=float('inf'); resume_train_totals=[]; resume_task_totals={}; resume_train_seconds=0.0
if LAST.exists():
    ckpt=torch.load(LAST,map_location=DEVICE,weights_only=False)
    model.load_state_dict(ckpt['model']); objective.load_state_dict(ckpt['objective']); optimizer.load_state_dict(ckpt['optimizer']); scheduler.load_state_dict(ckpt['scheduler']); scaler.load_state_dict(ckpt['scaler'])
    start_epoch=ckpt['epoch']; resume_batch=ckpt.get('batch_in_epoch',0); resume_order=ckpt.get('epoch_order'); best_score=ckpt.get('best_score',float('inf')); history=ckpt.get('history',[]); resume_train_totals=ckpt.get('partial_train_totals',[]); resume_task_totals=ckpt.get('partial_task_totals',{}); resume_train_seconds=ckpt.get('train_seconds_elapsed',0.0)
    random.setstate(ckpt['python_rng']); np.random.set_state(ckpt['numpy_rng']); torch.set_rng_state(ckpt['torch_rng'])
    if torch.cuda.is_available() and ckpt.get('cuda_rng') is not None: torch.cuda.set_rng_state_all(ckpt['cuda_rng'])
    print(f'Resuming epoch {start_epoch+1}, after {resume_batch} completed batches.')

for epoch in range(start_epoch,EPOCHS):
    if DEVICE.type=='cuda': torch.cuda.synchronize()
    epoch_start=time.perf_counter()
    if DEVICE.type=='cuda': torch.cuda.reset_peak_memory_stats(DEVICE)
    model.train(); objective.train()
    if epoch==start_epoch and resume_batch>0:
        train_totals=list(resume_train_totals); task_totals={k:list(v) for k,v in resume_task_totals.items()}
    else:
        train_totals=[]; task_totals={'depth_loss':[],'normal_loss':[]}
    if epoch==start_epoch and resume_order is not None:
        epoch_order=resume_order
    else:
        if DATA_IO_MODE == 'drive_shards':
            epoch_order=shard_sample_order(datasets['train'],SEED+epoch)
        elif DATA_IO_MODE in ('drive_fuse', 'drive_api'):
            block_size=1024
            blocks=[list(range(start,min(start+block_size,len(datasets['train'])))) for start in range(0,len(datasets['train']),block_size)]
            random.Random(SEED+epoch).shuffle(blocks)
            epoch_order=[idx for block in blocks for idx in block]
        else:
            epoch_order=torch.randperm(len(datasets['train']),generator=torch.Generator().manual_seed(SEED+epoch)).tolist()
        resume_batch=0
    start_sample=resume_batch*BATCH_SIZE
    remaining_indices=epoch_order[start_sample:]
    epoch_loader=DataLoader(datasets['train'],batch_size=BATCH_SIZE,sampler=remaining_indices,num_workers=NUM_WORKERS,pin_memory=(DEVICE.type=='cuda'),drop_last=False,generator=torch.Generator().manual_seed(SEED+10000+epoch))
    prior_train_seconds=resume_train_seconds if epoch==start_epoch else 0.0
    last_checkpoint_time=time.monotonic()
    last_batch_end=time.perf_counter()
    session_data_wait_seconds=0.0
    from tqdm.auto import tqdm
    full_batches=(len(datasets['train'])+BATCH_SIZE-1)//BATCH_SIZE
    progress_bar=tqdm(epoch_loader,total=full_batches,initial=resume_batch,desc=f'Epoch {epoch+1}/{EPOCHS}',unit='batch',dynamic_ncols=True)
    for batch_idx,(x,d,n,m) in enumerate(progress_bar,start=resume_batch):
        data_wait_seconds=time.perf_counter()-last_batch_end
        session_data_wait_seconds+=data_wait_seconds
        x,d,n,m=[z.to(DEVICE,non_blocking=True) for z in (x,d,n,m)]
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type,dtype=(torch.float16 if DEVICE.type=='cuda' else torch.bfloat16),enabled=(DEVICE.type=='cuda')):
            pd,pn=model(x); loss,parts=objective(pd,d,pn,n,m)
        if not torch.isfinite(loss): raise FloatingPointError(f'Non-finite training loss at epoch {epoch}')
        scaler.scale(loss).backward(); scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(list(model.parameters())+list(objective.parameters()),1.0); scaler.step(optimizer); scaler.update()
        train_totals.append(float(loss.detach()))
        for k,v in parts.items(): task_totals[k].append(float(v))
        completed_batches=batch_idx+1
        mem_gib=(torch.cuda.memory_allocated(DEVICE)/1024**3) if DEVICE.type=='cuda' else 0.0
        progress_bar.set_postfix(iteration=f'{completed_batches}/{full_batches}',loss=f'{float(loss.detach()):.4f}',depth=f'{float(parts.get("depth_loss", torch.tensor(float("nan")))):.4f}',normal=f'{float(parts.get("normal_loss", torch.tensor(float("nan")))):.4f}',gpu_mem=f'{mem_gib:.1f}GiB',shards=SHARD_STATS['copies'],local_hits=SHARD_STATS['local_hits'],data_s=f'{data_wait_seconds:.2f}',refresh=False)
        if completed_batches%PROGRESS_EVERY_STEPS==0:
            progress={'epoch':epoch+1,'epochs_total':EPOCHS,'batches_complete':completed_batches,'batches_total':(len(datasets['train'])+BATCH_SIZE-1)//BATCH_SIZE,'latest_train_loss':float(loss.detach()),'last_update_utc':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime())}
            progress_tmp=PROGRESS_PATH.with_name(PROGRESS_PATH.name+'.tmp'); progress_tmp.write_text(json.dumps(progress,indent=2)); progress_tmp.replace(PROGRESS_PATH)
            print(f"epoch {epoch+1:03d}/{EPOCHS} batch {completed_batches}/{progress['batches_total']} loss={float(loss.detach()):.4f}")
        if time.monotonic()-last_checkpoint_time>=CHECKPOINT_EVERY_MINUTES*60:
            if DEVICE.type=='cuda': torch.cuda.synchronize()
            save_checkpoint(checkpoint_state(epoch,completed_batches,epoch_order,best_score,train_totals,task_totals,prior_train_seconds+time.perf_counter()-epoch_start),LAST)
            last_checkpoint_time=time.monotonic()
            print(f'Saved resumable checkpoint after epoch {epoch+1}, batch {completed_batches}.')
        last_batch_end=time.perf_counter()
    if DEVICE.type=='cuda': torch.cuda.synchronize()
    train_seconds=prior_train_seconds+time.perf_counter()-epoch_start
    val_start=time.perf_counter()
    val_metrics=evaluate(model,loaders['val'])
    if DEVICE.type=='cuda': torch.cuda.synchronize()
    val_seconds=time.perf_counter()-val_start
    preview_dir=OUTPUT_DIR/'epoch_outputs'; preview_dir.mkdir(parents=True,exist_ok=True)
    preview_x,preview_d,preview_n,preview_m=next(iter(loaders['val']))
    model.eval()
    with torch.inference_mode():
        preview_pd,preview_pn=model(preview_x[:1].to(DEVICE))
    # Persist one reproducible per-epoch matrix sample (depth + XYZ normals + GT + mask); full predictions are represented by aggregate metrics.
    preview_path=preview_dir/f'epoch_{epoch+1:03d}_sample.npz'
    np.savez_compressed(preview_path,
        depth_pred_m=preview_pd[0,0].float().cpu().numpy(),
        normal_pred_xyz=preview_pn[0].float().cpu().numpy(),
        depth_gt_m=preview_d[0,0].numpy(),
        normal_gt_xyz=preview_n[0].numpy(),
        valid_mask=preview_m[0,0].numpy())
    output_summary={'depth_pred_min_m':float(preview_pd.min()),'depth_pred_median_m':float(preview_pd.median()),'depth_pred_max_m':float(preview_pd.max()),'normal_pred_norm_mean':float(torch.linalg.vector_norm(preview_pn,dim=1).mean()),'matrix_file':str(preview_path)}
    epoch_seconds=time.perf_counter()-epoch_start
    samples_per_second=len(datasets['train'])/max(train_seconds,1e-6)
    eta_seconds=max(EPOCHS-epoch-1,0)*epoch_seconds
    score=(val_metrics['depth_rmse_m']/0.1)+(val_metrics['normal_mean_deg']/30.0)
    scheduler.step(score)
    peak_gpu_memory_gib=(torch.cuda.max_memory_allocated(DEVICE)/1024**3) if DEVICE.type=='cuda' else 0.0
    record={'epoch':epoch+1,'session_data_wait_seconds':session_data_wait_seconds,'peak_gpu_memory_gib':peak_gpu_memory_gib,'train_loss':float(np.mean(train_totals)),**{k:float(np.mean(v)) for k,v in task_totals.items()},'val':val_metrics,'selection_score':score,'train_seconds':train_seconds,'validation_seconds':val_seconds,'epoch_seconds':epoch_seconds,'train_samples_per_second':samples_per_second,'eta_seconds':eta_seconds,'lr':optimizer.param_groups[0]['lr'],'task_sigmas':torch.exp(objective.log_vars.detach()).cpu().tolist(),'output_summary':output_summary,'steps_per_epoch':full_batches,'train_samples':len(datasets['train']),'validation_samples':len(datasets['val'])}
    history.append(record)
    history_tmp=HISTORY_PATH.with_name(HISTORY_PATH.name+'.tmp'); history_tmp.write_text(json.dumps(history,indent=2)); history_tmp.replace(HISTORY_PATH)
    completed_state=checkpoint_state(epoch+1,0,None,min(best_score,score),[],{},0.0)
    save_checkpoint(completed_state,LAST)
    if score<best_score:
        best_score=score; completed_state['best_score']=best_score; completed_state['best_epoch']=epoch+1; save_checkpoint(completed_state,CHECKPOINT)
    progress={'epoch':epoch+1,'epochs_total':EPOCHS,'status':'epoch_complete','train_seconds':train_seconds,'validation_seconds':val_seconds,'epoch_seconds':epoch_seconds,'train_samples_per_second':samples_per_second,'eta_seconds':eta_seconds,'preview_matrix':str(preview_path)}
    progress_tmp=PROGRESS_PATH.with_name(PROGRESS_PATH.name+'.tmp'); progress_tmp.write_text(json.dumps(progress,indent=2)); progress_tmp.replace(PROGRESS_PATH)
    resume_batch=0; resume_order=None; resume_train_totals=[]; resume_task_totals={}; resume_train_seconds=0.0
    print(f"epoch {epoch+1:03d}/{EPOCHS} iterations={full_batches:,} train={record['train_loss']:.4f} val depth(RMSE m)={val_metrics['depth_rmse_m']:.4f} normal(mean°)={val_metrics['normal_mean_deg']:.2f} score={score:.4f} train={train_seconds/60:.1f}m val={val_seconds/60:.1f}m rate={samples_per_second:.1f} samples/s ETA={eta_seconds/60:.1f}m")

if not CHECKPOINT.exists() and LAST.exists():
    save_checkpoint(torch.load(LAST,map_location=DEVICE,weights_only=False),CHECKPOINT)
if not CHECKPOINT.exists(): raise RuntimeError('No checkpoint found. Complete at least one epoch first.')
best=torch.load(CHECKPOINT,map_location=DEVICE,weights_only=False); model.load_state_dict(best['model'])
print('Best checkpoint epoch:',best.get('best_epoch',best['epoch']),'selection score:',best['best_score'])

### Reading training progress

50,000 samples split into 35,000 train / 7,500 validation / 7,500 test. Batch size 8 means 4,375 training iterations per epoch and 21,875 over five epochs. The live bar reports losses, CUDA memory, staged shard count, local read hits, and data_s (time waiting for the next batch). Archive transfers appear as occasional stalls; most sample reads within a staged archive are local. Epoch order shuffles archives and samples inside each archive. Preparation and training timings are separate; measure actual batches before estimating total training time. Checkpoints resume the saved sample order and batch position. After a runtime reset, the archives remain on Drive and the bounded local cache is rebuilt.

In [ ]:
# 11. Locked test evaluation and qualitative depth/normal inspection
test_metrics=evaluate(model,loaders['test'])
print(json.dumps(test_metrics,indent=2))
metrics_path=OUTPUT_DIR/'test_metrics.json'; metrics_tmp=metrics_path.with_name(metrics_path.name+'.tmp')
metrics_tmp.write_text(json.dumps(test_metrics,indent=2)); metrics_tmp.replace(metrics_path)
model.eval(); x,d,n,m=next(iter(loaders['test'])); x=x.to(DEVICE)
with torch.inference_mode(): pd,pn=model(x)
rgb=((x[0].cpu()*torch.tensor([.229,.224,.225])[:,None,None])+torch.tensor([.485,.456,.406])[:,None,None]).permute(1,2,0).clamp(0,1)
fig,ax=plt.subplots(2,3,figsize=(12,7))
ax[0,0].imshow(rgb); ax[0,0].set_title('Input RGB')
ax[0,1].imshow(d[0,0],cmap='viridis'); ax[0,1].set_title('Depth GT (m)')
ax[0,2].imshow(n[0].permute(1,2,0)*.5+.5); ax[0,2].set_title('Normals GT')
ax[1,0].imshow(pd[0,0].cpu(),cmap='viridis'); ax[1,0].set_title('Depth prediction (m)')
ax[1,1].imshow(pn[0].cpu().permute(1,2,0)*.5+.5); ax[1,1].set_title('Normals prediction')
ax[1,2].imshow(m[0,0],cmap='gray'); ax[1,2].set_title('Valid mask')
[z.axis('off') for z in ax.flat]; plt.tight_layout(); plt.show()


In [ ]:
# 12. Optional fixed-shape Core ML export; run with a pinned coremltools version.
# Conversion success is not a performance result. Follow with Xcode performance report on target iPhone(s).
EXPORT_COREML=False
if EXPORT_COREML:
    import coremltools as ct
    class ExportWrapper(nn.Module):
        def __init__(self,base): super().__init__(); self.base=base
        def forward(self,x): return self.base(x)
    export_model=ExportWrapper(model.eval()).cpu().eval()
    example=torch.zeros(1,3,*INPUT_SIZE)
    traced=torch.jit.trace(export_model,example,strict=True)
    # The model takes ImageNet-normalized RGB tensors; export as tensor input and normalize in app preprocessing.
    mlmodel=ct.convert(traced,source='pytorch',inputs=[ct.TensorType(name='image',shape=example.shape)],outputs=[ct.TensorType(name='depth'),ct.TensorType(name='normals')],convert_to='mlprogram',minimum_deployment_target=ct.target.iOS16,compute_precision=ct.precision.FLOAT16)
    mlmodel.author='Apple Capstone Portrait Relighting'; mlmodel.short_description='Portrait depth and surface-normal research model; evaluate on device before use.'
    out=OUTPUT_DIR/'portrait_geometry_depth_normals.mlpackage'; mlmodel.save(str(out)); print('Saved',out)
    # Numerical parity check on a representative fixed-shape tensor.
    if platform.system() == 'Darwin':
        with torch.inference_mode(): torch_out=export_model(example)
        coreml_out=mlmodel.predict({'image':example.numpy()})
        for name,ref in zip(('depth','normals'),torch_out):
            delta=np.abs(coreml_out[name]-ref.numpy())
            print(name,'max_abs_error',float(delta.max()),'mean_abs_error',float(delta.mean()))
            if not np.isfinite(delta).all(): raise FloatingPointError(f'Non-finite Core ML output for {name}')
    else:
        print('Export completed. Core ML prediction/parity check requires macOS; run that check on a Mac or target device.')

# Device evaluation checklist:
# 1) Xcode Model Performance Report: operation placement (ANE/GPU/CPU), load and prediction duration.
# 2) On each target iPhone/iOS: warm-up; >=200 predictions; report p50/p95, cold load, package size, peak memory.
# 3) Measure Vision segmentation + preprocessing + model + relighting + compositing frame time; include thermal steady state.
# 4) Compare FP32/FP16/quantized candidates for parity, quality, model size, and device latency.
# 5) 60 FPS requires the complete pipeline to sustain <16.7 ms/frame with acceptable quality and thermal behavior.

## Interpretation and required experiments
## Backbone evidence and decision
- **Keep FastViT-T8 as the baseline.** Apple reports FastViT-T8 at 3.6M parameters and provides Core ML classification measurements on iPhone models (for example 0.52 ms on iPhone 16 Pro). The FastViT paper and official model page support mobile efficiency and Core ML availability. Those figures are for the backbone/classifier model, not our 256×256 two-output dense model.
- **MobileNetV3-Large is a useful ablation.** Its paper targets mobile CPU efficiency, reports about 5.4M parameters, and introduces LR-ASPP for dense prediction. Its published CPU measurements use Pixel devices; they do not prove better Apple Neural Engine performance.
- **These published latency values are not directly comparable.** The devices, runtime, input graph, precision, and task differ. Change only the backbone while holding the dataset split, FPN/heads, resolution, training schedule, and Core ML conversion constant. Select based on held-out task quality, model size, and Xcode measurements on the intended iPhone.

- First establish that label overlays and coordinate conventions are correct. Pseudo-label metrics quantify agreement with the teacher, not necessarily physical truth.
- Ablate: (A) FastViT-T8 baseline with compact FPN, (B) MobileNetV3-Large comparison using the same FPN/heads, (C) fixed versus uncertainty-weighted losses. Change only the backbone for the backbone ablation; compare validation metrics, parameter count, converted model size, and on-device latency. Published backbone-only classification timings are not comparable to this dense prediction graph. - Compare against task-appropriate baselines: teacher outputs for geometry, constant/median depth where meaningful, 
- For the product claim (relighting), build a controlled capture set with fixed subject, camera/exposure/white balance, and measured light positions/intensities. Hold out identities and lighting configurations. Evaluate relit RGB against captures and inspect temporal stability on video.
- References: [FastViT paper](https://arxiv.org/abs/2303.14189); [Sapiens paper](https://arxiv.org/abs/2408.12569); [uncertainty weighting](https://arxiv.org/abs/1705.07115); [Core ML performance report](https://developer.apple.com/documentation/coreml/analyzing-a-core-ml-model-s-performance-in-xcode); [Core ML Tools PyTorch conversion](https://apple.github.io/coremltools/docs-guides/source/convert-pytorch.html); [MobileNetV3 paper](https://openaccess.thecvf.com/content_ICCV_2019/html/Howard_Searching_for_MobileNetV3_ICCV_2019_paper.html).
## Dataset trial scope
- The automatic Colab trial uses the official Microsoft SynthHuman/DAViD download utility. Its synthetic sample labels include RGB, absolute depth in centimetres, XYZ normals, and alpha masks; the active experiment supervises depth and normals only.
- The official release offers a single-sample smoke test and a single chunk (up to 5,000 samples; up to 8.75 GB); the full dataset is about 330 GB. The trial split uses sample IDs because the distributed files do not expose identity IDs, so it is not an identity-disjoint generalization test.
